In [ ]:
# ============================================================
# RECOVERY CELL — run this after any disconnect, before resuming work
# ============================================================
!pip install -q transformers scikit-learn shap accelerate

import torch, pandas as pd, numpy as np
from pathlib import Path
from google.colab import drive
from transformers import AutoTokenizer, AutoModel
from torchvision import models

drive.mount('/content/drive')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", DEVICE)

PROJECT_ROOT = Path("/content/drive/MyDrive/Explainable-Multimodal-Fake-News")
DATA_ROOT = PROJECT_ROOT / "data"
FEATURE_DIR = DATA_ROOT / "features"
SUBSET_DIR = DATA_ROOT / "multimodal" / "subset"
V2_ROOT = DATA_ROOT / "v2_restart"
V2_MODELS, V2_RESULTS = V2_ROOT / "models", V2_ROOT / "results"

LABEL_MAP = {0: "FAKE", 1: "REAL"}
CLASS_NAMES = ["FAKE", "REAL"]

# Reload frozen features
train_txt = torch.load(FEATURE_DIR / "train_muril_features.pt", map_location="cpu").float()
test_txt  = torch.load(FEATURE_DIR / "test_muril_features.pt", map_location="cpu").float()
val_txt   = torch.load(FEATURE_DIR / "validation_muril_features.pt", map_location="cpu").float()
test_y    = torch.load(FEATURE_DIR / "test_muril_labels.pt", map_location="cpu").long()
val_y     = torch.load(FEATURE_DIR / "validation_muril_labels.pt", map_location="cpu").long()
train_img = torch.load(FEATURE_DIR / "train_image_features.pt", map_location="cpu")["features"].float()
test_img  = torch.load(FEATURE_DIR / "test_image_features.pt", map_location="cpu")["features"].float()
val_img   = torch.load(FEATURE_DIR / "validation_image_features.pt", map_location="cpu")["features"].float()

# Reload normalization stats (NOT recomputed — must match training exactly)
norm = torch.load(V2_MODELS / "v2_normalization_stats.pt", map_location=DEVICE)
text_mean, text_std = norm["text_mean"].to(DEVICE), norm["text_std"].to(DEVICE)
image_mean, image_std = norm["image_mean"].to(DEVICE), norm["image_std"].to(DEVICE)

test_txt_n = (test_txt.to(DEVICE) - text_mean) / text_std
test_img_n = (test_img.to(DEVICE) - image_mean) / image_std
val_txt_n  = (val_txt.to(DEVICE) - text_mean) / text_std
val_img_n  = (val_img.to(DEVICE) - image_mean) / image_std

# Rebuild model architecture, then load trained weights
import torch.nn as nn
class UnifiedMultimodalClassifier(nn.Module):
    def __init__(self, text_dim=768, image_dim=2048, hidden_dim=256, num_classes=2, dropout=0.4):
        super().__init__()
        self.text_branch = nn.Sequential(nn.Linear(text_dim,512), nn.ReLU(), nn.Dropout(dropout), nn.Linear(512,hidden_dim), nn.ReLU())
        self.image_branch = nn.Sequential(nn.Linear(image_dim,512), nn.ReLU(), nn.Dropout(dropout), nn.Linear(512,hidden_dim), nn.ReLU())
        self.classifier = nn.Sequential(nn.Linear(hidden_dim*2,256), nn.ReLU(), nn.Dropout(dropout), nn.Linear(256,num_classes))
    def forward(self, t, i):
        return self.classifier(torch.cat([self.text_branch(t), self.image_branch(i)], dim=1))

unified_model = UnifiedMultimodalClassifier().to(DEVICE)
checkpoint = torch.load(V2_MODELS / "v2_unified_model_best.pt", map_location=DEVICE)
unified_model.load_state_dict(checkpoint["model_state_dict"])
unified_model.eval()
print(f"✓ Model restored — val F1 was {checkpoint['val_f1']:.4f}")

# Reload MuRIL for live text encoding (needed for SHAP / general inference)
muril_tokenizer = AutoTokenizer.from_pretrained("google/muril-base-cased")
muril_model = AutoModel.from_pretrained("google/muril-base-cased").to(DEVICE)
muril_model.eval()

test_df = pd.read_csv(SUBSET_DIR / "test_subset.csv")
full_test_df = pd.read_csv(DATA_ROOT / "multimodal" / "test_multimodal.csv")
print("✓ Fully recovered — ready to resume.")

Mounted at /content/drive
Device: cuda
✓ Model restored — val F1 was 0.8356


config.json:   0%|          | 0.00/411 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/206 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/3.16M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/113 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: google/muril-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors: reconstructing file:   0%|          |  0.00B /  953MB            

model.safetensors: downloading bytes:           |  0.00B            

✓ Fully recovered — ready to resume.


In [ ]:
!pip install -q transformers scikit-learn shap google-api-python-client accelerate

import torch, time
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")
if DEVICE.type == 'cuda':
    print("GPU:", torch.cuda.get_device_name(0))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using device: cuda
GPU: Tesla T4


In [ ]:
# ============================================================
# SEED — run this every time, right after Setup, before anything else
# ============================================================
import random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print(f"✓ Random seed fixed at {SEED} — training is now reproducible")

✓ Random seed fixed at 42 — training is now reproducible


In [ ]:
PROJECT_ROOT = Path("/content/drive/MyDrive/Explainable-Multimodal-Fake-News")
DATA_ROOT    = PROJECT_ROOT / "data"

FEATURE_DIR = DATA_ROOT / "features"          # OLD — reused as-is, read-only

V2_ROOT    = DATA_ROOT / "v2_restart"          # NEW — everything below writes here
V2_CACHE   = V2_ROOT / "cache"
V2_MODELS  = V2_ROOT / "models"
V2_RESULTS = V2_ROOT / "results"
for d in [V2_CACHE, V2_MODELS, V2_RESULTS]:
    d.mkdir(parents=True, exist_ok=True)

LABEL_MAP   = {0: "FAKE", 1: "REAL"}
CLASS_NAMES = ["FAKE", "REAL"]

print("Reading features from:", FEATURE_DIR)
print("Writing new outputs to:", V2_ROOT)

Reading features from: /content/drive/MyDrive/Explainable-Multimodal-Fake-News/data/features
Writing new outputs to: /content/drive/MyDrive/Explainable-Multimodal-Fake-News/data/v2_restart


In [ ]:
print("Loading saved MuRIL text features...")
train_txt = torch.load(FEATURE_DIR / "train_muril_features.pt", map_location="cpu").float()
val_txt   = torch.load(FEATURE_DIR / "validation_muril_features.pt", map_location="cpu").float()
test_txt  = torch.load(FEATURE_DIR / "test_muril_features.pt", map_location="cpu").float()

train_y = torch.load(FEATURE_DIR / "train_muril_labels.pt", map_location="cpu").long()
val_y   = torch.load(FEATURE_DIR / "validation_muril_labels.pt", map_location="cpu").long()
test_y  = torch.load(FEATURE_DIR / "test_muril_labels.pt", map_location="cpu").long()

print("Loading saved ResNet-50 image features...")
train_img_raw = torch.load(FEATURE_DIR / "train_image_features.pt", map_location="cpu")
val_img_raw   = torch.load(FEATURE_DIR / "validation_image_features.pt", map_location="cpu")
test_img_raw  = torch.load(FEATURE_DIR / "test_image_features.pt", map_location="cpu")

train_img = train_img_raw["features"].float()
val_img   = val_img_raw["features"].float()
test_img  = test_img_raw["features"].float()

print("\nShapes — text/image/labels:")
print("Train:", train_txt.shape, train_img.shape, train_y.shape)
print("Val  :", val_txt.shape, val_img.shape, val_y.shape)
print("Test :", test_txt.shape, test_img.shape, test_y.shape)

assert train_txt.shape[0] == train_img.shape[0] == train_y.shape[0]
assert val_txt.shape[0]   == val_img.shape[0]   == val_y.shape[0]
assert test_txt.shape[0]  == test_img.shape[0]  == test_y.shape[0]

# Cross-check: image-pipeline labels vs MuRIL-pipeline labels (past runs had order bugs here)
for name, img_raw, muril_y in [("train", train_img_raw, train_y), ("val", val_img_raw, val_y), ("test", test_img_raw, test_y)]:
    match = torch.equal(img_raw["labels"].long(), muril_y)
    print(f"{name}: text/image label match = {match}" + ("" if match else "  ⚠ MISMATCH — investigate before training"))

print("\n✓ Alignment verified")

Loading saved MuRIL text features...
Loading saved ResNet-50 image features...

Shapes — text/image/labels:
Train: torch.Size([10000, 768]) torch.Size([10000, 2048]) torch.Size([10000])
Val  : torch.Size([2000, 768]) torch.Size([2000, 2048]) torch.Size([2000])
Test : torch.Size([2000, 768]) torch.Size([2000, 2048]) torch.Size([2000])
train: text/image label match = True
val: text/image label match = True
test: text/image label match = True

✓ Alignment verified


In [ ]:
def compute_stats(x):
    mean = x.mean(dim=0, keepdim=True)
    std = x.std(dim=0, keepdim=True).clamp(min=1e-6)
    return mean, std

text_mean, text_std = compute_stats(train_txt)
image_mean, image_std = compute_stats(train_img)

norm = lambda x, m, s: (x - m) / s
train_txt_n, val_txt_n, test_txt_n = [norm(x, text_mean, text_std) for x in (train_txt, val_txt, test_txt)]
train_img_n, val_img_n, test_img_n = [norm(x, image_mean, image_std) for x in (train_img, val_img, test_img)]

torch.save({"text_mean": text_mean, "text_std": text_std,
            "image_mean": image_mean, "image_std": image_std},
           V2_MODELS / "v2_normalization_stats.pt")
print("✓ Normalized. Stats saved to", V2_MODELS / "v2_normalization_stats.pt")

✓ Normalized. Stats saved to /content/drive/MyDrive/Explainable-Multimodal-Fake-News/data/v2_restart/models/v2_normalization_stats.pt


In [ ]:
from torch.utils.data import Dataset, DataLoader

class UnifiedFeatureDataset(Dataset):
    def __init__(self, text_feats, image_feats, labels, modality_dropout_p=0.0):
        self.text_feats, self.image_feats, self.labels = text_feats, image_feats, labels
        self.modality_dropout_p = modality_dropout_p

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        text = self.text_feats[idx].clone()
        image = self.image_feats[idx].clone()
        label = self.labels[idx]
        if self.modality_dropout_p > 0:
            r = torch.rand(1).item()
            if r < self.modality_dropout_p / 2:
                text = torch.zeros_like(text)
            elif r < self.modality_dropout_p:
                image = torch.zeros_like(image)
        return text, image, label

MODALITY_DROPOUT_P = 0.3
BATCH_SIZE = 128

train_dataset = UnifiedFeatureDataset(train_txt_n, train_img_n, train_y, modality_dropout_p=MODALITY_DROPOUT_P)
val_dataset   = UnifiedFeatureDataset(val_txt_n, val_img_n, val_y, modality_dropout_p=0.0)
test_dataset  = UnifiedFeatureDataset(test_txt_n, test_img_n, test_y, modality_dropout_p=0.0)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("Batches — train/val/test:", len(train_loader), len(val_loader), len(test_loader))

Batches — train/val/test: 79 16 16


In [ ]:
import torch.nn as nn

class UnifiedMultimodalClassifier(nn.Module):
    def __init__(self, text_dim=768, image_dim=2048, hidden_dim=256, num_classes=2, dropout=0.4):
        super().__init__()
        self.text_branch = nn.Sequential(
            nn.Linear(text_dim, 512), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(512, hidden_dim), nn.ReLU())
        self.image_branch = nn.Sequential(
            nn.Linear(image_dim, 512), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(512, hidden_dim), nn.ReLU())
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, 256), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256, num_classes))

    def forward(self, text_features, image_features):
        fused = torch.cat([self.text_branch(text_features), self.image_branch(image_features)], dim=1)
        return self.classifier(fused)

unified_model = UnifiedMultimodalClassifier().to(DEVICE)
print(unified_model)
print(f"\nTotal parameters: {sum(p.numel() for p in unified_model.parameters()):,}")

UnifiedMultimodalClassifier(
  (text_branch): Sequential(
    (0): Linear(in_features=768, out_features=512, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.4, inplace=False)
    (3): Linear(in_features=512, out_features=256, bias=True)
    (4): ReLU()
  )
  (image_branch): Sequential(
    (0): Linear(in_features=2048, out_features=512, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.4, inplace=False)
    (3): Linear(in_features=512, out_features=256, bias=True)
    (4): ReLU()
  )
  (classifier): Sequential(
    (0): Linear(in_features=512, out_features=256, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.4, inplace=False)
    (3): Linear(in_features=256, out_features=2, bias=True)
  )
)

Total parameters: 1,837,314


In [ ]:
import torch.optim as optim
from sklearn.metrics import f1_score

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(unified_model.parameters(), lr=5e-4, weight_decay=1e-3)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)

NUM_EPOCHS, PATIENCE = 25, 6
BEST_MODEL_PATH = V2_MODELS / "v2_unified_model_best.pt"

def run_epoch(loader, train_mode):
    unified_model.train() if train_mode else unified_model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []
    with torch.set_grad_enabled(train_mode):
        for text_f, image_f, labels in loader:
            text_f, image_f, labels = text_f.to(DEVICE), image_f.to(DEVICE), labels.to(DEVICE)
            if train_mode: optimizer.zero_grad()
            logits = unified_model(text_f, image_f)
            loss = criterion(logits, labels)
            if train_mode:
                loss.backward(); optimizer.step()
            total_loss += loss.item() * labels.size(0)
            preds = torch.argmax(logits, dim=1)
            correct += (preds == labels).sum().item(); total += labels.size(0)
            all_preds.extend(preds.cpu().numpy()); all_labels.extend(labels.cpu().numpy())
    return total_loss/total, correct/total, all_preds, all_labels

best_val_f1, epochs_no_improve = 0.0, 0
start = time.time()
for epoch in range(NUM_EPOCHS):
    train_loss, train_acc, _, _ = run_epoch(train_loader, True)
    val_loss, val_acc, val_preds, val_labels = run_epoch(val_loader, False)
    val_f1 = f1_score(val_labels, val_preds, average="macro")
    scheduler.step(val_f1)
    print(f"Epoch {epoch+1:2d}/{NUM_EPOCHS} | train loss {train_loss:.4f} acc {train_acc:.4f} | "
          f"val loss {val_loss:.4f} acc {val_acc:.4f} f1 {val_f1:.4f}")
    if val_f1 > best_val_f1:
        best_val_f1, epochs_no_improve = val_f1, 0
        torch.save({"epoch": epoch, "model_state_dict": unified_model.state_dict(),
                    "val_loss": val_loss, "val_accuracy": val_acc, "val_f1": val_f1}, BEST_MODEL_PATH)
        print(f"  ✓ new best saved (val F1 {val_f1:.4f})")
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f"\nEarly stopping at epoch {epoch+1}"); break

print(f"\nDone in {(time.time()-start)/60:.1f} min. Best val F1: {best_val_f1:.4f}")

Epoch  1/25 | train loss 0.4702 acc 0.7754 | val loss 0.3722 acc 0.8270 f1 0.8266
  ✓ new best saved (val F1 0.8266)
Epoch  2/25 | train loss 0.3518 acc 0.8479 | val loss 0.3727 acc 0.8335 f1 0.8330
  ✓ new best saved (val F1 0.8330)
Epoch  3/25 | train loss 0.2866 acc 0.8748 | val loss 0.3782 acc 0.8360 f1 0.8356
  ✓ new best saved (val F1 0.8356)
Epoch  4/25 | train loss 0.2416 acc 0.9013 | val loss 0.4081 acc 0.8275 f1 0.8275
Epoch  5/25 | train loss 0.2023 acc 0.9161 | val loss 0.4345 acc 0.8325 f1 0.8324
Epoch  6/25 | train loss 0.1811 acc 0.9233 | val loss 0.4996 acc 0.8345 f1 0.8345
Epoch  7/25 | train loss 0.1485 acc 0.9398 | val loss 0.5312 acc 0.8285 f1 0.8283
Epoch  8/25 | train loss 0.1181 acc 0.9521 | val loss 0.5853 acc 0.8215 f1 0.8215
Epoch  9/25 | train loss 0.0925 acc 0.9629 | val loss 0.6613 acc 0.8255 f1 0.8254

Early stopping at epoch 9

Done in 0.3 min. Best val F1: 0.8356


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
import pandas as pd

checkpoint = torch.load(BEST_MODEL_PATH, map_location=DEVICE)
unified_model.load_state_dict(checkpoint["model_state_dict"])
unified_model.eval()
print(f"Loaded checkpoint — epoch {checkpoint['epoch']}, val F1 {checkpoint['val_f1']:.4f}")

@torch.no_grad()
def evaluate_mode(text_feats, image_feats, labels, mode):
    text_feats, image_feats = text_feats.clone().to(DEVICE), image_feats.clone().to(DEVICE)
    if mode == "text_only":
        image_feats = torch.zeros_like(image_feats)
    elif mode == "image_only":
        text_feats = torch.zeros_like(text_feats)
    probs = torch.softmax(unified_model(text_feats, image_feats), dim=1)
    preds = torch.argmax(probs, dim=1).cpu().numpy()
    y = labels.numpy()
    metrics = {"Mode": mode, "Accuracy": accuracy_score(y, preds),
               "Precision": precision_score(y, preds, zero_division=0),
               "Recall": recall_score(y, preds, zero_division=0),
               "F1": f1_score(y, preds, zero_division=0),
               "ROC-AUC": roc_auc_score(y, probs[:, 1].cpu().numpy())}
    return metrics, confusion_matrix(y, preds)

print("=" * 70); print("ABLATION: ONE MODEL, THREE INPUT MODES"); print("=" * 70)
results = []
for mode in ["multimodal", "text_only", "image_only"]:
    metrics, cm = evaluate_mode(test_txt_n, test_img_n, test_y, mode)
    results.append(metrics)
    print(f"\n{mode.upper()}"); [print(f"  {k}: {v:.4f}") for k, v in metrics.items() if k != "Mode"]
    print("  Confusion matrix:\n", cm)

results_df = pd.DataFrame(results)
results_df.to_csv(V2_RESULTS / "v2_ablation_results.csv", index=False)
print("\n", results_df.to_string(index=False))
print("\n✓ Saved to", V2_RESULTS / "v2_ablation_results.csv")

Loaded checkpoint — epoch 2, val F1 0.8356
ABLATION: ONE MODEL, THREE INPUT MODES

MULTIMODAL
  Accuracy: 0.8625
  Precision: 0.8292
  Recall: 0.9130
  F1: 0.8691
  ROC-AUC: 0.9281
  Confusion matrix:
 [[812 188]
 [ 87 913]]

TEXT_ONLY
  Accuracy: 0.8100
  Precision: 0.8051
  Recall: 0.8180
  F1: 0.8115
  ROC-AUC: 0.8898
  Confusion matrix:
 [[802 198]
 [182 818]]

IMAGE_ONLY
  Accuracy: 0.7705
  Precision: 0.7310
  Recall: 0.8560
  F1: 0.7886
  ROC-AUC: 0.8446
  Confusion matrix:
 [[685 315]
 [144 856]]

       Mode  Accuracy  Precision  Recall       F1  ROC-AUC
multimodal    0.8625   0.829246   0.913 0.869110 0.928148
 text_only    0.8100   0.805118   0.818 0.811508 0.889787
image_only    0.7705   0.730999   0.856 0.788577 0.844577

✓ Saved to /content/drive/MyDrive/Explainable-Multimodal-Fake-News/data/v2_restart/results/v2_ablation_results.csv


In [ ]:
FAKE_WEIGHT, REAL_WEIGHT = 1.3, 1.0  # penalize missed FAKE more

class_weights = torch.tensor([FAKE_WEIGHT, REAL_WEIGHT]).to(DEVICE)
criterion_weighted = nn.CrossEntropyLoss(weight=class_weights)

unified_model_weighted = UnifiedMultimodalClassifier().to(DEVICE)
optimizer = optim.AdamW(unified_model_weighted.parameters(), lr=5e-4, weight_decay=1e-3)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)
BEST_WEIGHTED_PATH = V2_MODELS / "v2_unified_model_weighted_best.pt"

def run_epoch_weighted(loader, train_mode, model, crit):
    model.train() if train_mode else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []
    with torch.set_grad_enabled(train_mode):
        for text_f, image_f, labels in loader:
            text_f, image_f, labels = text_f.to(DEVICE), image_f.to(DEVICE), labels.to(DEVICE)
            if train_mode: optimizer.zero_grad()
            logits = model(text_f, image_f)
            loss = crit(logits, labels)
            if train_mode:
                loss.backward(); optimizer.step()
            total_loss += loss.item() * labels.size(0)
            preds = torch.argmax(logits, dim=1)
            correct += (preds == labels).sum().item(); total += labels.size(0)
            all_preds.extend(preds.cpu().numpy()); all_labels.extend(labels.cpu().numpy())
    return total_loss/total, correct/total, all_preds, all_labels

best_val_f1 = 0.0
for epoch in range(25):
    train_loss, train_acc, _, _ = run_epoch_weighted(train_loader, True, unified_model_weighted, criterion_weighted)
    val_loss, val_acc, val_preds, val_labels = run_epoch_weighted(val_loader, False, unified_model_weighted, criterion_weighted)
    val_f1 = f1_score(val_labels, val_preds, average="macro")
    scheduler.step(val_f1)
    print(f"Epoch {epoch+1:2d} | train acc {train_acc:.4f} | val acc {val_acc:.4f} f1 {val_f1:.4f}")
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save({"model_state_dict": unified_model_weighted.state_dict(), "val_f1": val_f1}, BEST_WEIGHTED_PATH)
        print("  ✓ new best saved")

# Compare against the unweighted model on test set
checkpoint_w = torch.load(BEST_WEIGHTED_PATH, map_location=DEVICE)
unified_model_weighted.load_state_dict(checkpoint_w["model_state_dict"])
unified_model_weighted.eval()

@torch.no_grad()
def evaluate_weighted(model, mode):
    text_feats, image_feats = test_txt_n.clone().to(DEVICE), test_img_n.clone().to(DEVICE)
    if mode == "text_only": image_feats = torch.zeros_like(image_feats)
    elif mode == "image_only": text_feats = torch.zeros_like(text_feats)
    probs = torch.softmax(model(text_feats, image_feats), dim=1)
    preds = torch.argmax(probs, dim=1).cpu().numpy()
    y = test_y.numpy()
    return (accuracy_score(y, preds), f1_score(y, preds, zero_division=0),
            recall_score(y, preds, pos_label=0, zero_division=0),  # FAKE recall
            confusion_matrix(y, preds))

print("\n" + "="*60 + "\nWEIGHTED MODEL — MULTIMODAL TEST RESULTS\n" + "="*60)
acc, f1, fake_recall, cm = evaluate_weighted(unified_model_weighted, "multimodal")
print(f"Accuracy: {acc:.4f} | F1: {f1:.4f} | FAKE recall: {fake_recall:.4f}\n{cm}")

Epoch  1 | train acc 0.7512 | val acc 0.8240 f1 0.8240
  ✓ new best saved
Epoch  2 | train acc 0.8484 | val acc 0.8350 f1 0.8350
  ✓ new best saved
Epoch  3 | train acc 0.8758 | val acc 0.8260 f1 0.8256
Epoch  4 | train acc 0.9031 | val acc 0.8320 f1 0.8320
Epoch  5 | train acc 0.9207 | val acc 0.8385 f1 0.8384
  ✓ new best saved
Epoch  6 | train acc 0.9309 | val acc 0.8220 f1 0.8219
Epoch  7 | train acc 0.9359 | val acc 0.8305 f1 0.8301
Epoch  8 | train acc 0.9447 | val acc 0.8160 f1 0.8157
Epoch  9 | train acc 0.9498 | val acc 0.8195 f1 0.8195
Epoch 10 | train acc 0.9602 | val acc 0.8360 f1 0.8359
Epoch 11 | train acc 0.9672 | val acc 0.8310 f1 0.8310
Epoch 12 | train acc 0.9688 | val acc 0.8235 f1 0.8235
Epoch 13 | train acc 0.9742 | val acc 0.8230 f1 0.8230
Epoch 14 | train acc 0.9763 | val acc 0.8310 f1 0.8310
Epoch 15 | train acc 0.9789 | val acc 0.8315 f1 0.8315
Epoch 16 | train acc 0.9770 | val acc 0.8240 f1 0.8240
Epoch 17 | train acc 0.9807 | val acc 0.8285 f1 0.8285
Epoch 18

In [ ]:
from sklearn.metrics import f1_score
import numpy as np

@torch.no_grad()
def get_val_probs(model):
    logits = model(val_txt_n.to(DEVICE), val_img_n.to(DEVICE))
    return torch.softmax(logits, dim=1)[:, 1].cpu().numpy()  # P(REAL)

val_probs = get_val_probs(unified_model)  # or unified_model_weighted, whichever you keep
thresholds = np.arange(0.30, 0.71, 0.02)
best_thresh, best_f1 = 0.5, 0
for t in thresholds:
    preds = (val_probs >= t).astype(int)
    f1 = f1_score(val_y.numpy(), preds, average="macro")
    if f1 > best_f1:
        best_f1, best_thresh = f1, t

print(f"Best threshold on validation: {best_thresh:.2f} (macro F1 {best_f1:.4f})")

@torch.no_grad()
def evaluate_with_threshold(model, threshold, mode="multimodal"):
    text_feats, image_feats = test_txt_n.clone().to(DEVICE), test_img_n.clone().to(DEVICE)
    if mode == "text_only": image_feats = torch.zeros_like(image_feats)
    elif mode == "image_only": text_feats = torch.zeros_like(text_feats)
    probs = torch.softmax(model(text_feats, image_feats), dim=1)[:, 1].cpu().numpy()
    preds = (probs >= threshold).astype(int)
    y = test_y.numpy()
    return accuracy_score(y, preds), f1_score(y, preds, average="macro"), confusion_matrix(y, preds)

acc, f1, cm = evaluate_with_threshold(unified_model, best_thresh, "multimodal")
print(f"\nWith tuned threshold — Accuracy: {acc:.4f} | Macro F1: {f1:.4f}\n{cm}")

Best threshold on validation: 0.48 (macro F1 0.8360)

With tuned threshold — Accuracy: 0.8600 | Macro F1: 0.8595
[[803 197]
 [ 83 917]]


In [ ]:
# ============================================================
# CELL 11 — PER-MODE THRESHOLD CALIBRATION + FINAL ABLATION TABLE
# Run after Cell 8 (requires: unified_model loaded with best
# checkpoint, and test_txt_n/test_img_n/test_y, val_txt_n/val_img_n/val_y)
# ============================================================

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, recall_score, roc_auc_score, confusion_matrix

@torch.no_grad()
def get_probs(model, text_feats, image_feats, mode):
    text_feats, image_feats = text_feats.clone().to(DEVICE), image_feats.clone().to(DEVICE)
    if mode == "text_only":
        image_feats = torch.zeros_like(image_feats)
    elif mode == "image_only":
        text_feats = torch.zeros_like(text_feats)
    logits = model(text_feats, image_feats)
    return torch.softmax(logits, dim=1)[:, 1].cpu().numpy()  # P(REAL)


def tune_threshold(model, mode):
    probs = get_probs(model, val_txt_n, val_img_n, mode)
    y = val_y.numpy()
    best_t, best_f1 = 0.5, 0.0
    for t in np.arange(0.30, 0.71, 0.02):
        preds = (probs >= t).astype(int)
        f1 = f1_score(y, preds, average="macro")
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return best_t, best_f1


print("=" * 70)
print("PER-MODE THRESHOLD CALIBRATION (tuned on validation set)")
print("=" * 70)

final_results = []
for mode in ["multimodal", "text_only", "image_only"]:
    thresh, val_f1 = tune_threshold(unified_model, mode)

    test_probs = get_probs(unified_model, test_txt_n, test_img_n, mode)
    test_preds = (test_probs >= thresh).astype(int)
    y_true = test_y.numpy()

    metrics = {
        "Mode": mode,
        "Threshold": thresh,
        "Accuracy": accuracy_score(y_true, test_preds),
        "Macro F1": f1_score(y_true, test_preds, average="macro"),
        "FAKE Recall": recall_score(y_true, test_preds, pos_label=0, zero_division=0),
        "REAL Recall": recall_score(y_true, test_preds, pos_label=1, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, test_probs),
    }
    cm = confusion_matrix(y_true, test_preds)

    print(f"\n{mode.upper()}  (threshold = {thresh:.2f}, tuned val macro F1 = {val_f1:.4f})")
    for k, v in metrics.items():
        if k not in ("Mode", "Threshold"):
            print(f"  {k}: {v:.4f}")
    print("  Confusion matrix:\n", cm)

    final_results.append(metrics)

final_df = pd.DataFrame(final_results)
final_df.to_csv(V2_RESULTS / "v2_final_calibrated_ablation.csv", index=False)

print("\n" + "=" * 70)
print("FINAL CALIBRATED ABLATION TABLE — use this one for the report")
print("=" * 70)
print(final_df.to_string(index=False))
print("\n✓ Saved to", V2_RESULTS / "v2_final_calibrated_ablation.csv")

PER-MODE THRESHOLD CALIBRATION (tuned on validation set)

MULTIMODAL  (threshold = 0.48, tuned val macro F1 = 0.8360)
  Accuracy: 0.8600
  Macro F1: 0.8595
  FAKE Recall: 0.8030
  REAL Recall: 0.9170
  ROC-AUC: 0.9281
  Confusion matrix:
 [[803 197]
 [ 83 917]]

TEXT_ONLY  (threshold = 0.46, tuned val macro F1 = 0.7920)
  Accuracy: 0.8085
  Macro F1: 0.8083
  FAKE Recall: 0.7790
  REAL Recall: 0.8380
  ROC-AUC: 0.8898
  Confusion matrix:
 [[779 221]
 [162 838]]

IMAGE_ONLY  (threshold = 0.56, tuned val macro F1 = 0.7602)
  Accuracy: 0.7725
  Macro F1: 0.7720
  FAKE Recall: 0.7270
  REAL Recall: 0.8180
  ROC-AUC: 0.8446
  Confusion matrix:
 [[727 273]
 [182 818]]

FINAL CALIBRATED ABLATION TABLE — use this one for the report
      Mode  Threshold  Accuracy  Macro F1  FAKE Recall  REAL Recall  ROC-AUC
multimodal       0.48    0.8600  0.859544        0.803        0.917 0.928148
 text_only       0.46    0.8085  0.808333        0.779        0.838 0.889787
image_only       0.56    0.7725  0.

In [ ]:
from transformers import AutoTokenizer, AutoModel
import pandas as pd

MURIL_NAME = "google/muril-base-cased"
muril_tokenizer = AutoTokenizer.from_pretrained(MURIL_NAME)
muril_model = AutoModel.from_pretrained(MURIL_NAME).to(DEVICE)
muril_model.eval()

SUBSET_DIR = DATA_ROOT / "multimodal" / "subset"
test_df = pd.read_csv(SUBSET_DIR / "test_subset.csv")
print("Test rows:", len(test_df))
print(test_df[["id", "text", "label"]].head(3))

# Reload normalization stats in case this is a fresh session
norm_stats = torch.load(V2_MODELS / "v2_normalization_stats.pt", map_location=DEVICE)
text_mean, text_std = norm_stats["text_mean"].to(DEVICE), norm_stats["text_std"].to(DEVICE)
image_mean, image_std = norm_stats["image_mean"].to(DEVICE), norm_stats["image_std"].to(DEVICE)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: google/muril-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Test rows: 2000
        id                                               text  label
0   420uzp  park police finally removed that snow penis so...      1
1  ewp9f70                              privates investigator      0
2   dh3pjt                              conjoined grape twins      1


In [ ]:
import numpy as np
import shap

@torch.no_grad()
def predict_proba_from_text(texts, fixed_image_norm):
    texts = [str(t) if t else "" for t in texts]
    encoded = muril_tokenizer(texts, padding=True, truncation=True, max_length=256, return_tensors="pt").to(DEVICE)
    outputs = muril_model(**encoded)
    token_embeddings = outputs.last_hidden_state
    attention_mask = encoded["attention_mask"]
    mask = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    summed = torch.sum(token_embeddings * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    text_emb = summed / counts
    text_emb_n = (text_emb - text_mean.to(DEVICE)) / text_std.to(DEVICE)   # <- fixed

    img_batch = fixed_image_norm.unsqueeze(0).repeat(len(texts), 1).to(DEVICE)
    logits = unified_model(text_emb_n, img_batch)
    probs = torch.softmax(logits, dim=1)
    return probs.cpu().numpy()

masker = shap.maskers.Text(tokenizer=r"\W+")   # split on whitespace/punctuation -> word-level, not subword-level
print("✓ Text masker ready (word-level, not subword)")

✓ Text masker ready (word-level, not subword)


In [ ]:
DEMO_INDICES = [0, 1, 2]   # pick a few interesting ones after you see results — e.g. one correct, one wrong

for idx in DEMO_INDICES:
    row = test_df.iloc[idx]
    text = str(row["text"])
    true_label = LABEL_MAP[int(row["label"])]
    fixed_image = test_img_n[idx]

    explainer = shap.Explainer(lambda x: predict_proba_from_text(x, fixed_image), masker, output_names=CLASS_NAMES)
    shap_values = explainer([text])

    pred_probs = predict_proba_from_text([text], fixed_image)[0]
    pred_label = CLASS_NAMES[int(np.argmax(pred_probs))]

    print("=" * 70)
    print(f"Sample {idx} | True: {true_label} | Predicted: {pred_label} ({pred_probs[np.argmax(pred_probs)]:.2%} confidence)")
    print("-" * 70)

    tokens = shap_values.data[0]
    values = shap_values.values[0][:, int(np.argmax(pred_probs))]   # contribution toward the predicted class
    ranked = sorted(zip(tokens, values), key=lambda x: -abs(x[1]))

    print("Top contributing words:")
    for word, val in ranked[:10]:
        direction = "→ pushes toward " + pred_label if val > 0 else "→ pushes away from " + pred_label
        print(f"  {word.strip():20s}  {val:+.4f}   {direction}")
    print()

# Save one full example for the slide deck
import json
demo_row = test_df.iloc[DEMO_INDICES[0]]
with open(V2_RESULTS / "v2_token_shap_demo_sample0.json", "w") as f:
    json.dump({"text": demo_row["text"], "true_label": LABEL_MAP[int(demo_row["label"])],
               "tokens": list(shap_values.data[0]), "values": shap_values.values[0].tolist()}, f, ensure_ascii=False, indent=2)
print("✓ Saved demo sample for slides")

Sample 0 | True: REAL | Predicted: REAL (99.94% confidence)
----------------------------------------------------------------------
Top contributing words:
  bigger                +0.1342   → pushes toward REAL
  an                    +0.1330   → pushes toward REAL
  penis                 +0.1318   → pushes toward REAL
  snow                  +0.1249   → pushes toward REAL
  one                   +0.1218   → pushes toward REAL
  park                  +0.0965   → pushes toward REAL
  even                  +0.0690   → pushes toward REAL
  someone               -0.0548   → pushes away from REAL
  with                  +0.0494   → pushes toward REAL
  police                +0.0488   → pushes toward REAL

Sample 1 | True: FAKE | Predicted: REAL (95.28% confidence)
----------------------------------------------------------------------
Top contributing words:
  investigator          +0.9442   → pushes toward REAL
  privates              +0.0086   → pushes toward REAL

Sample 2 | True: REAL | P

In [ ]:
# ============================================================
# CELL 17 — GENERAL INFERENCE: handles text-only, image-only,
# or both, on ANY raw text/image, not just precomputed test rows
# ============================================================
from torchvision import models
from PIL import Image

resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
image_encoder = torch.nn.Sequential(*list(resnet.children())[:-1]).to(DEVICE)
image_encoder.eval()
image_transform = models.ResNet50_Weights.DEFAULT.transforms()

@torch.no_grad()
def encode_text(text):
    encoded = muril_tokenizer([text], padding=True, truncation=True, max_length=256, return_tensors="pt").to(DEVICE)
    outputs = muril_model(**encoded)
    token_embeddings = outputs.last_hidden_state
    attention_mask = encoded["attention_mask"]
    mask = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    summed = torch.sum(token_embeddings * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    emb = summed / counts
    return (emb - text_mean.to(DEVICE)) / text_std.to(DEVICE)   # <- fixed

@torch.no_grad()
def encode_image(image_path):
    img = Image.open(image_path).convert("RGB")
    img = image_transform(img).unsqueeze(0).to(DEVICE)
    feat = image_encoder(img).flatten(1)
    return (feat - image_mean.to(DEVICE)) / image_std.to(DEVICE)   # <- fixed

@torch.no_grad()
def predict(text=None, image_path=None):
    """Accepts text only, image only, or both. Returns prediction + confidence + mode used."""
    if text is None and image_path is None:
        raise ValueError("Provide at least text or an image_path")

    text_vec = encode_text(text) if text else torch.zeros(1, 768).to(DEVICE)
    image_vec = encode_image(image_path) if image_path else torch.zeros(1, 2048).to(DEVICE)

    mode = "multimodal" if (text and image_path) else ("text_only" if text else "image_only")
    probs = torch.softmax(unified_model(text_vec, image_vec), dim=1)[0]

    # apply the per-mode calibrated threshold from your final ablation run
    thresholds = {"multimodal": 0.60, "text_only": 0.54, "image_only": 0.56}
    pred = 1 if probs[1].item() >= thresholds[mode] else 0

    return {"mode": mode, "prediction": CLASS_NAMES[pred], "confidence": probs[pred].item(),
            "fake_prob": probs[0].item(), "real_prob": probs[1].item()}

# Quick tests — one row from your test CSV, exercising all three modes
row = test_df.iloc[0]
img_path = DATA_ROOT / "images" / "test" / f"{row['id']}.jpg"

print("Text + image:", predict(text=row["text"], image_path=img_path))
print("Text only   :", predict(text=row["text"]))
print("Image only  :", predict(image_path=img_path))
print("True label  :", LABEL_MAP[int(row["label"])])

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 179MB/s]


Text + image: {'mode': 'multimodal', 'prediction': 'REAL', 'confidence': 0.9544718861579895, 'fake_prob': 0.04552807658910751, 'real_prob': 0.9544718861579895}
Text only   : {'mode': 'text_only', 'prediction': 'REAL', 'confidence': 0.9236812591552734, 'fake_prob': 0.07631880044937134, 'real_prob': 0.9236812591552734}
Image only  : {'mode': 'image_only', 'prediction': 'REAL', 'confidence': 0.7113197445869446, 'fake_prob': 0.2886802852153778, 'real_prob': 0.7113197445869446}
True label  : REAL


In [ ]:
# Sanity check: does live encoding match the precomputed features for the same row?
row = test_df.iloc[5]
img_path = DATA_ROOT / "images" / "test" / f"{row['id']}.jpg"

live_result = predict(text=row["text"], image_path=img_path)

precomputed_probs = torch.softmax(
    unified_model(test_txt_n[5:6].to(DEVICE), test_img_n[5:6].to(DEVICE)), dim=1
)[0]

print("Live encoder  :", live_result)
print("Precomputed   : fake_prob={:.4f} real_prob={:.4f}".format(
    precomputed_probs[0].item(), precomputed_probs[1].item()))
print("True label    :", LABEL_MAP[int(row['label'])])

Live encoder  : {'mode': 'multimodal', 'prediction': 'REAL', 'confidence': 0.999980092048645, 'fake_prob': 1.9914750737370923e-05, 'real_prob': 0.999980092048645}
Precomputed   : fake_prob=0.0000 real_prob=1.0000
True label    : REAL


In [ ]:
import requests

FACT_CHECK_API_KEY = "YOUR_FACT_CHECK_API_KEY"   # <- paste your key

def fact_check_search(query, api_key=FACT_CHECK_API_KEY, max_results=3):
    url = "https://factchecktools.googleapis.com/v1alpha1/claims:search"   # <- fixed
    params = {"query": query[:200], "key": api_key, "pageSize": max_results}
    try:
        resp = requests.get(url, params=params, timeout=15)
        resp.raise_for_status()
        claims = resp.json().get("claims", [])[:max_results]
    except Exception as e:
        print("Fact Check API error:", e)
        return []

    results = []
    for c in claims:
        review = (c.get("claimReview") or [{}])[0]
        results.append({
            "claim_text": c.get("text", ""),
            "claimant": c.get("claimant", "Unknown"),
            "rating": review.get("textualRating", "N/A"),
            "publisher": review.get("publisher", {}).get("name", "Unknown"),
            "url": review.get("url", ""),
        })
    return results

# Quick test
test_query = test_df.iloc[5]["text"][:150]
print("Query:", test_query)
results = fact_check_search(test_query)
if results:
    for r in results:
        print(f"\n  [{r['rating']}] {r['publisher']}\n  {r['claim_text']}\n  {r['url']}")
else:
    print("\n  No matches found (expected for most articles — coverage is limited to claims that have actually been fact-checked).")

Query: germany says taking photos of food infringes the chefs copyright

  No matches found (expected for most articles — coverage is limited to claims that have actually been fact-checked).


In [ ]:
def full_pipeline_demo(idx):
    row = test_df.iloc[idx]
    text, true_label = str(row["text"]), LABEL_MAP[int(row["label"])]
    img_path = DATA_ROOT / "images" / "test" / f"{row['id']}.jpg"

    result = predict(text=text, image_path=img_path)

    fixed_image = test_img_n[idx]
    explainer = shap.Explainer(
        lambda x: predict_proba_from_text(x, fixed_image),
        masker,
        output_names=CLASS_NAMES,
        algorithm="permutation",   # <- avoids the clustering crash on short text
    )
    shap_values = explainer([text])
    tokens = shap_values.data[0]
    pred_class_idx = CLASS_NAMES.index(result["prediction"])
    values = shap_values.values[0][:, pred_class_idx]
    top_words = sorted(zip(tokens, values), key=lambda x: -abs(x[1]))[:5]

    evidence = fact_check_search(text[:150])

    print("=" * 70); print(f"FINAL OUTPUT — Sample {idx}"); print("=" * 70)
    print(f"Prediction: {result['prediction']}  (confidence {result['confidence']:.2%})")
    print(f"True label: {true_label}")
    print("\nTop words behind this decision:")
    for w, v in top_words:
        print(f"  {w.strip():15s} {v:+.4f}")
    print("\nSupporting evidence:")
    if evidence:
        for e in evidence:
            print(f"  [{e['rating']}] {e['publisher']} — {e['url']}")
    else:
        print("  No fact-check matches found for this article.")
    print()

for idx in [10, 13, 21]:
    try:
        full_pipeline_demo(idx)
    except Exception as e:
        print(f"⚠ Sample {idx} failed: {e}\n")

⚠ Sample 10 failed: name 'predict' is not defined

⚠ Sample 13 failed: name 'predict' is not defined

⚠ Sample 21 failed: name 'predict' is not defined



In [ ]:
# Scan a batch of test rows for ones with real Fact Check API coverage
FOUND_WITH_EVIDENCE = []

for idx in range(0, 50):   # scan the first 50 test rows
    row = test_df.iloc[idx]
    query = str(row["text"])[:150]
    matches = fact_check_search(query)
    if matches:
        FOUND_WITH_EVIDENCE.append(idx)
        print(f"idx {idx}: FOUND {len(matches)} match(es) — \"{query[:60]}...\"")

print(f"\nTotal rows with evidence found: {len(FOUND_WITH_EVIDENCE)} out of 50 checked")
print("Usable demo indices:", FOUND_WITH_EVIDENCE)

idx 10: FOUND 3 match(es) — "karate..."
idx 13: FOUND 3 match(es) — "wat..."
idx 21: FOUND 1 match(es) — "surprised owl..."

Total rows with evidence found: 3 out of 50 checked
Usable demo indices: [10, 13, 21]


In [ ]:
import pandas as pd
pd.read_csv(V2_RESULTS / "v2_final_calibrated_ablation.csv")

,Mode,Threshold,Accuracy,Macro F1,FAKE Recall,REAL Recall,ROC-AUC
0,multimodal,0.50,0.8455,0.845326,0.812,0.879,0.921462
1,text_only,0.54,0.8090,0.808993,0.803,0.815,0.884457
2,image_only,0.34,0.7675,0.764678,0.658,0.877,0.826268


In [ ]:
import re
for idx in [10, 13, 21]:
    text = str(test_df.iloc[idx]["text"])
    n_words = len(re.findall(r"\w+", text))
    print(f"idx {idx}: {n_words} words | preview: {text[:100]!r}")

idx 10: 1 words | preview: 'karate'
idx 13: 1 words | preview: 'wat'
idx 21: 2 words | preview: 'surprised owl'


In [ ]:
import re

def full_pipeline_demo(idx):
    row = test_df.iloc[idx]
    text, true_label = str(row["text"]), LABEL_MAP[int(row["label"])]
    img_path = DATA_ROOT / "images" / "test" / f"{row['id']}.jpg"

    result = predict(text=text, image_path=img_path)
    fixed_image = test_img_n[idx]

    print("=" * 70); print(f"FINAL OUTPUT — Sample {idx}"); print("=" * 70)
    print(f"Prediction: {result['prediction']}  (confidence {result['confidence']:.2%})")
    print(f"True label: {true_label}")

    print("\nTop words behind this decision:")
    try:
        n_words = max(len(re.findall(r"\w+", text)), 1)
        max_evals = max(2 * n_words + 1, 500)   # SHAP's own minimum requirement

        explainer = shap.Explainer(
            lambda x: predict_proba_from_text(x, fixed_image),
            masker, output_names=CLASS_NAMES, algorithm="permutation",
        )
        shap_values = explainer([text], max_evals=max_evals)
        tokens = shap_values.data[0]
        pred_class_idx = CLASS_NAMES.index(result["prediction"])
        values = shap_values.values[0][:, pred_class_idx]
        top_words = sorted(zip(tokens, values), key=lambda x: -abs(x[1]))[:5]
        for w, v in top_words:
            print(f"  {w.strip():15s} {v:+.4f}")
    except Exception as e:
        print(f"  (word-level explanation unavailable for this sample: {e})")

    evidence = fact_check_search(text[:150])
    print("\nSupporting evidence:")
    if evidence:
        for e in evidence:
            print(f"  [{e['rating']}] {e['publisher']} — {e['url']}")
    else:
        print("  No fact-check matches found for this article.")
    print()

full_pipeline_demo(21)
print("\n\n")
full_pipeline_demo(49)

FINAL OUTPUT — Sample 21
Prediction: REAL  (confidence 77.54%)
True label: REAL

Top words behind this decision:
  (word-level explanation unavailable for this sample: name 'shap' is not defined)

Supporting evidence:
  No fact-check matches found for this article.




FINAL OUTPUT — Sample 49
Prediction: REAL  (confidence 91.69%)
True label: REAL

Top words behind this decision:
  (word-level explanation unavailable for this sample: name 'shap' is not defined)

Supporting evidence:
  No fact-check matches found for this article.



In [ ]:
%%writefile app.py
import streamlit as st
import torch
import torch.nn as nn
import numpy as np
import re
import requests
from pathlib import Path
from PIL import Image
from transformers import AutoTokenizer, AutoModel
from torchvision import models
import shap

st.set_page_config(page_title="Explainable Multilingual Fake News Detection", layout="wide")
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

PROJECT_ROOT = Path("/content/drive/MyDrive/Explainable-Multimodal-Fake-News")
DATA_ROOT = PROJECT_ROOT / "data"
V2_MODELS = DATA_ROOT / "v2_restart" / "models"

CLASS_NAMES = ["FAKE", "REAL"]
THRESHOLDS = {"multimodal": 0.48, "text_only": 0.46, "image_only": 0.56}
FACT_CHECK_API_KEY = "YOUR_FACT_CHECK_API_KEY"   # <- paste your key (same one as your notebook)

class UnifiedMultimodalClassifier(nn.Module):
    def __init__(self, text_dim=768, image_dim=2048, hidden_dim=256, num_classes=2, dropout=0.4):
        super().__init__()
        self.text_branch = nn.Sequential(nn.Linear(text_dim,512), nn.ReLU(), nn.Dropout(dropout), nn.Linear(512,hidden_dim), nn.ReLU())
        self.image_branch = nn.Sequential(nn.Linear(image_dim,512), nn.ReLU(), nn.Dropout(dropout), nn.Linear(512,hidden_dim), nn.ReLU())
        self.classifier = nn.Sequential(nn.Linear(hidden_dim*2,256), nn.ReLU(), nn.Dropout(dropout), nn.Linear(256,num_classes))
    def forward(self, t, i):
        return self.classifier(torch.cat([self.text_branch(t), self.image_branch(i)], dim=1))

@st.cache_resource
def load_everything():
    norm = torch.load(V2_MODELS / "v2_normalization_stats.pt", map_location=DEVICE)
    text_mean, text_std = norm["text_mean"].to(DEVICE), norm["text_std"].to(DEVICE)
    image_mean, image_std = norm["image_mean"].to(DEVICE), norm["image_std"].to(DEVICE)

    model = UnifiedMultimodalClassifier().to(DEVICE)
    ckpt = torch.load(V2_MODELS / "v2_unified_model_best.pt", map_location=DEVICE)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()

    tokenizer = AutoTokenizer.from_pretrained("google/muril-base-cased")
    text_model = AutoModel.from_pretrained("google/muril-base-cased").to(DEVICE)
    text_model.eval()

    resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
    image_encoder = nn.Sequential(*list(resnet.children())[:-1]).to(DEVICE)
    image_encoder.eval()
    image_transform = models.ResNet50_Weights.DEFAULT.transforms()

    masker = shap.maskers.Text(tokenizer=r"\W+")
    return model, tokenizer, text_model, image_encoder, image_transform, masker, text_mean, text_std, image_mean, image_std

model, tokenizer, text_model, image_encoder, image_transform, masker, text_mean, text_std, image_mean, image_std = load_everything()

@torch.no_grad()
def encode_text(text):
    encoded = tokenizer([text], padding=True, truncation=True, max_length=256, return_tensors="pt").to(DEVICE)
    out = text_model(**encoded)
    mask = encoded["attention_mask"].unsqueeze(-1).expand(out.last_hidden_state.size()).float()
    emb = (out.last_hidden_state * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
    return (emb - text_mean) / text_std

@torch.no_grad()
def encode_image(pil_image):
    img = image_transform(pil_image.convert("RGB")).unsqueeze(0).to(DEVICE)
    feat = image_encoder(img).flatten(1)
    return (feat - image_mean) / image_std

@torch.no_grad()
def predict(text=None, pil_image=None):
    text_vec = encode_text(text) if text else torch.zeros(1, 768).to(DEVICE)
    image_vec = encode_image(pil_image) if pil_image is not None else torch.zeros(1, 2048).to(DEVICE)
    mode = "multimodal" if (text and pil_image is not None) else ("text_only" if text else "image_only")
    probs = torch.softmax(model(text_vec, image_vec), dim=1)[0]
    pred = 1 if probs[1].item() >= THRESHOLDS[mode] else 0
    return {"mode": mode, "prediction": CLASS_NAMES[pred], "confidence": probs[pred].item(),
            "fake_prob": probs[0].item(), "real_prob": probs[1].item()}

@torch.no_grad()
def predict_proba_from_text(texts, fixed_image_vec):
    texts = [str(t) if t else "" for t in texts]
    encoded = tokenizer(texts, padding=True, truncation=True, max_length=256, return_tensors="pt").to(DEVICE)
    out = text_model(**encoded)
    mask = encoded["attention_mask"].unsqueeze(-1).expand(out.last_hidden_state.size()).float()
    text_emb = (out.last_hidden_state * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
    text_emb = (text_emb - text_mean) / text_std
    img_batch = fixed_image_vec.repeat(len(texts), 1)
    probs = torch.softmax(model(text_emb, img_batch), dim=1)
    return probs.cpu().numpy()

def explain_text(text, image_vec, prediction):
    try:
        n_words = max(len(re.findall(r"\w+", text)), 1)
        max_evals = max(2 * n_words + 1, 500)
        explainer = shap.Explainer(lambda x: predict_proba_from_text(x, image_vec), masker,
                                    output_names=CLASS_NAMES, algorithm="permutation")
        shap_values = explainer([text], max_evals=max_evals)
        pred_idx = CLASS_NAMES.index(prediction)
        values = shap_values.values[0][:, pred_idx]
        return sorted(zip(shap_values.data[0], values), key=lambda x: -abs(x[1]))[:8]
    except Exception:
        return None

def fact_check_search(query, max_results=3):
    url = "https://factchecktools.googleapis.com/v1alpha1/claims:search"
    params = {"query": query[:200], "key": FACT_CHECK_API_KEY, "pageSize": max_results}
    try:
        resp = requests.get(url, params=params, timeout=15)
        resp.raise_for_status()
        claims = resp.json().get("claims", [])[:max_results]
    except Exception:
        return []
    out = []
    for c in claims:
        review = (c.get("claimReview") or [{}])[0]
        out.append({"rating": review.get("textualRating", "N/A"),
                     "publisher": review.get("publisher", {}).get("name", "Unknown"),
                     "url": review.get("url", "")})
    return out

# ---------------- UI ----------------
st.title("📰 Explainable Multilingual Fake News Detection")
st.caption("MuRIL classification · SHAP word-level explainability · Google Fact Check evidence")

lang = st.radio("Article language", ["English", "Hindi (हिन्दी)"], horizontal=True)
if lang.startswith("Hindi"):
    st.info("MuRIL supports Hindi text natively, but this system's calibration and evaluation were performed on English data — treat Hindi predictions as exploratory.")

col1, col2 = st.columns(2)
with col1:
    text_input = st.text_area("News article text", height=200, placeholder="Paste a news article here...")
with col2:
    image_input = st.file_uploader("News image (optional)", type=["jpg", "jpeg", "png"])
    if image_input:
        st.image(image_input, caption="Uploaded image", use_container_width=True)

if st.button("Analyze", type="primary"):
    if not text_input and not image_input:
        st.warning("Please provide article text, an image, or both.")
    else:
        pil_img = Image.open(image_input) if image_input else None
        with st.spinner("Analyzing..."):
            result = predict(text=text_input if text_input else None, pil_image=pil_img)

        st.divider()
        badge = "🟢" if result["prediction"] == "REAL" else "🔴"
        st.subheader(f"{badge} Prediction: {result['prediction']}  ({result['confidence']:.1%} confidence)")
        st.caption(f"Mode used: {result['mode'].replace('_', ' ')}")

        c1, c2 = st.columns(2)
        c1.metric("FAKE probability", f"{result['fake_prob']:.1%}")
        c2.metric("REAL probability", f"{result['real_prob']:.1%}")

        if text_input:
            st.markdown("### 🔍 Word-level explanation (SHAP)")
            image_vec = encode_image(pil_img) if pil_img is not None else torch.zeros(1, 2048).to(DEVICE)
            top_words = explain_text(text_input, image_vec, result["prediction"])
            if top_words:
                for w, v in top_words:
                    color = "#2ecc71" if v > 0 else "#e74c3c"
                    direction = f"pushes toward {result['prediction']}" if v > 0 else f"pushes away from {result['prediction']}"
                    st.markdown(f"<div style='background:{color}22;border-left:4px solid {color};padding:6px 10px;margin:4px 0;'>"
                                f"<b>{w.strip()}</b> &nbsp; ({v:+.3f}) — {direction}</div>", unsafe_allow_html=True)
            else:
                st.info("Word-level explanation unavailable — text is too short to explain meaningfully.")

        st.markdown("### 📋 Fact Check evidence")
        if text_input:
            evidence = fact_check_search(text_input[:150])
            if evidence:
                for e in evidence:
                    st.markdown(f"**[{e['rating']}]** {e['publisher']} — [{e['url']}]({e['url']})")
            else:
                st.caption("No matching fact-check records found (expected for most articles — coverage is limited to previously reviewed claims).")
        else:
            st.caption("Fact-check evidence requires article text.")

Writing app.py


In [ ]:
!pip install -q streamlit
!npm install -g localtunnel --silent

import subprocess, time
subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"])
time.sleep(10)

print("Your tunnel password (public IP) is:")
!wget -q -O - ipv4.icanhazip.com
print("\nStarting tunnel — click the URL below once it appears:")
!npx localtunnel --port 8501

Your tunnel password (public IP) is:
35.204.198.135

Starting tunnel — click the URL below once it appears:
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙your url is: https://funny-squids-smile.loca.lt
^C


In [ ]:
!pkill -f streamlit
!pkill -f localtunnel

import subprocess, time

# Run Streamlit with output captured to a log file instead of silently discarded
log_file = open("streamlit_log.txt", "w")
subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"],
    stdout=log_file, stderr=subprocess.STDOUT
)

print("Waiting for Streamlit to fully start...")
time.sleep(45)   # give it real time to load MuRIL + ResNet-50

# Check whether it's actually listening on the port
!curl -s -o /dev/null -w "Local status: %{http_code}\n" http://localhost:8501

print("\n--- Last 30 lines of Streamlit's log ---")
!tail -n 30 streamlit_log.txt

Waiting for Streamlit to fully start...


KeyboardInterrupt: 

In [ ]:
!pkill -9 -f streamlit
!pkill -9 -f localtunnel
!pkill -9 -f "npx localtunnel"
import time
time.sleep(3)
print("All processes killed.")

All processes killed.


In [ ]:
import subprocess, time

log_file = open("streamlit_log.txt", "w")
subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"],
    stdout=log_file, stderr=subprocess.STDOUT
)

print("Waiting for Streamlit to fully start...")
time.sleep(45)

!curl -s -o /dev/null -w "Local status: %{http_code}\n" http://localhost:8501
print("\n--- Last 15 lines of log ---")
!tail -n 15 streamlit_log.txt

Waiting for Streamlit to fully start...
Local status: 200

--- Last 15 lines of log ---


2026-09-08 11:51:03.573 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.11.156.26:8501



In [ ]:
row = test_df.iloc[21]
print("TEXT:\n", row["text"])
print("\nIMAGE PATH:", DATA_ROOT / "images" / "test" / f"{row['id']}.jpg")
print("TRUE LABEL:", LABEL_MAP[int(row["label"])])

TEXT:
 surprised owl

IMAGE PATH: /content/drive/MyDrive/Explainable-Multimodal-Fake-News/data/images/test/b8adr3.jpg
TRUE LABEL: REAL


In [ ]:
demo_samples = [21, 49, 10, 13]

for idx in demo_samples:
    row = test_df.iloc[idx]
    print(f"\n{'='*70}")
    print(f"SAMPLE {idx}  |  True label: {LABEL_MAP[int(row['label'])]}")
    print(f"{'='*70}")
    print("TEXT:")
    print(row["text"])
    print(f"\nIMAGE FILE: {row['id']}.jpg")

# Copy the exact image files you'll need into one folder for easy download
import shutil
from pathlib import Path

DEMO_FOLDER = Path("/content/demo_images")
DEMO_FOLDER.mkdir(exist_ok=True)

for idx in demo_samples:
    row = test_df.iloc[idx]
    src = DATA_ROOT / "images" / "test" / f"{row['id']}.jpg"
    dst = DEMO_FOLDER / f"sample_{idx}.jpg"
    shutil.copy(src, dst)
    print(f"Copied sample {idx} -> {dst}")

# Zip them so you can download all at once
shutil.make_archive("/content/demo_images", "zip", DEMO_FOLDER)
print("\n✓ Download demo_images.zip from the Colab file browser (left sidebar)")


SAMPLE 21  |  True label: REAL
TEXT:
surprised owl

IMAGE FILE: b8adr3.jpg

SAMPLE 49  |  True label: REAL
TEXT:
israel to ally with arab neighbors around red sea in bid to save worlds corals

IMAGE FILE: c0t90i.jpg

SAMPLE 10  |  True label: FAKE
TEXT:
karate

IMAGE FILE: cl2065e.jpg

SAMPLE 13  |  True label: FAKE
TEXT:
wat

IMAGE FILE: c8lsls5.jpg
Copied sample 21 -> /content/demo_images/sample_21.jpg
Copied sample 49 -> /content/demo_images/sample_49.jpg
Copied sample 10 -> /content/demo_images/sample_10.jpg
Copied sample 13 -> /content/demo_images/sample_13.jpg

✓ Download demo_images.zip from the Colab file browser (left sidebar)


In [ ]:
import re

def first_sentence(text, max_chars=200):
    match = re.match(r'^(.*?[.!?])\s', text + ' ')
    sentence = match.group(1) if match else text[:max_chars]
    return sentence[:max_chars]

GOOD_CANDIDATES = []
for idx in range(0, 300):
    row = test_df.iloc[idx]
    text = str(row["text"])
    n_words = len(re.findall(r"\w+", text))
    if n_words < 12:          # skip short/odd fragments like "surprised owl"
        continue
    query = first_sentence(text)
    matches = fact_check_search(query)
    if matches:
        GOOD_CANDIDATES.append(idx)
        print(f"idx {idx} ({n_words} words): FOUND {len(matches)} — {query[:90]!r}")

print(f"\nTotal found: {len(GOOD_CANDIDATES)} out of 300 checked")
print(GOOD_CANDIDATES)


Total found: 0 out of 300 checked
[]


In [ ]:
import re
import pandas as pd

# Load the FULL multimodal pool, not just your balanced 2,000-row subset
full_test_df = pd.read_csv(DATA_ROOT / "multimodal" / "test_multimodal.csv")
print("Full test pool size:", len(full_test_df))

def first_sentence(text, max_chars=200):
    match = re.match(r'^(.*?[.!?])\s', str(text) + ' ')
    sentence = match.group(1) if match else str(text)[:max_chars]
    return sentence[:max_chars]

GOOD_CANDIDATES = []
for idx in range(0, len(full_test_df)):
    row = full_test_df.iloc[idx]
    text = str(row["text"])
    n_words = len(re.findall(r"\w+", text))
    if n_words < 12:
        continue
    query = first_sentence(text)
    matches = fact_check_search(query)
    if matches:
        GOOD_CANDIDATES.append((idx, row.get("id", idx)))
        print(f"row {idx} ({n_words} words): FOUND {len(matches)} — {query[:90]!r}")
    if len(GOOD_CANDIDATES) >= 6:   # stop once you have enough good options
        break

print(f"\nFound {len(GOOD_CANDIDATES)} candidates:", GOOD_CANDIDATES)

Full test pool size: 4728
row 153 (15 words): FOUND 1 — 'report on climate change shows canada warming at twice the rate of rest of world'
row 2349 (12 words): FOUND 3 — 'the clouds in this pic i took looks almost like a skull'
row 2872 (12 words): FOUND 1 — 'new vaccine to fight rotavirus a disease that kills children a day'
row 4311 (12 words): FOUND 1 — 'bear standing on a log trying to see outside of his cage'

Found 4 candidates: [(153, 'b8fmrn'), (2349, 'b5kj6i'), (2872, '6142vj'), (4311, '190w2q')]


In [ ]:
def full_pool_demo(df, idx):
    row = df.iloc[idx]
    text = str(row["text"])
    true_label = LABEL_MAP[int(row["label"])] if "label" in row and pd.notna(row["label"]) else "Unknown"
    img_path = DATA_ROOT / "images" / "test" / f"{row['id']}.jpg"

    if not img_path.exists():
        print(f"⚠ No image file found for id {row['id']} — running text-only")
        img_path = None

    result = predict(text=text, image_path=img_path)

    # live image encoding for SHAP's fixed_image (zeros if no image)
    if img_path:
        fixed_image = encode_image(img_path)[0]
    else:
        fixed_image = torch.zeros(2048).to(DEVICE)

    print("=" * 70); print(f"FULL POOL — row {idx} (id: {row['id']})"); print("=" * 70)
    print(f"Text: {text[:200]}...")
    print(f"\nPrediction: {result['prediction']}  (confidence {result['confidence']:.2%})")
    print(f"True label: {true_label}")

    print("\nTop words behind this decision:")
    try:
        n_words = max(len(re.findall(r"\w+", text)), 1)
        max_evals = max(2 * n_words + 1, 500)
        explainer = shap.Explainer(lambda x: predict_proba_from_text(x, fixed_image), masker,
                                    output_names=CLASS_NAMES, algorithm="permutation")
        shap_values = explainer([text], max_evals=max_evals)
        pred_idx = CLASS_NAMES.index(result["prediction"])
        values = shap_values.values[0][:, pred_idx]
        top_words = sorted(zip(shap_values.data[0], values), key=lambda x: -abs(x[1]))[:6]
        for w, v in top_words:
            print(f"  {w.strip():15s} {v:+.4f}")
    except Exception as e:
        print(f"  (unavailable: {e})")

    evidence = fact_check_search(first_sentence(text))
    print("\nSupporting evidence:")
    if evidence:
        for e in evidence:
            print(f"  [{e['rating']}] {e['publisher']} — {e['url']}")
    else:
        print("  No matches.")
    print()

for idx in [153, 2872]:
    full_pool_demo(full_test_df, idx)

FULL POOL — row 153 (id: b8fmrn)
Text: report on climate change shows canada warming at twice the rate of rest of world...

Prediction: REAL  (confidence 93.47%)
True label: REAL

Top words behind this decision:
  report          +0.1553
  shows           +0.1080
  climate         +0.1045
  at              +0.0887
  of              +0.0769
  rate            +0.0706

Supporting evidence:
  [Misleading] AFP Fact Check — https://factcheck.afp.com/doc.afp.com.362Q37C

FULL POOL — row 2872 (id: 6142vj)
Text: new vaccine to fight rotavirus a disease that kills children a day...

Prediction: REAL  (confidence 97.70%)
True label: REAL

Top words behind this decision:
  vaccine         +0.1938
  day             +0.1410
  children        +0.1299
  rotavirus       +0.1074
  a               +0.1038
  new             +0.0809

Supporting evidence:
  [Misleading] FactCheck.org — https://www.factcheck.org/2026/01/the-facts-on-the-vaccines-the-cdc-no-longer-recommends-for-all-kids/



In [ ]:
import shutil
row = full_test_df.iloc[153]  # or 153
shutil.copy(DATA_ROOT / "images" / "test" / f"{row['id']}.jpg", "/content/demo_extra1.jpg")

'/content/demo_extra1.jpg'

In [ ]:
print(repr(full_test_df.iloc[153]["text"]))

'report on climate change shows canada warming at twice the rate of rest of world'


In [ ]:
import pandas as pd
full_test_df = pd.read_csv(DATA_ROOT / "multimodal" / "test_multimodal.csv")
print("Full test pool size:", len(full_test_df))

Full test pool size: 4728


In [ ]:
import shutil
from pathlib import Path

DEMO_DIR = Path("/content/full_pool_demo_images")
DEMO_DIR.mkdir(exist_ok=True)
full_pool_rows = {
    "climate": 153,
    "vaccine": 2872,
    "skull_clouds": 2349,
}
for name, idx in full_pool_rows.items():
    row = full_test_df.iloc[idx]
    src = DATA_ROOT / "images" / "test" / f"{row['id']}.jpg"
    if src.exists():
        shutil.copy(src, DEMO_DIR / f"{name}.jpg")
        print(f"✓ {name}")
    else:
        print(f"✗ {name}: missing image, will need to run text-only for this one")

shutil.make_archive("/content/full_pool_demo_images", "zip", DEMO_DIR)
print("\nDownload full_pool_demo_images.zip from Colab's file browser")

✓ climate
✓ vaccine
✓ skull_clouds

Download full_pool_demo_images.zip from Colab's file browser


In [ ]:
hindi_wellknown_claims = [
    "व्हाट्सएप अब मैसेज भेजने के लिए पैसे चार्ज करेगा",              # WhatsApp will start charging for messages
    "चावल प्लास्टिक से बनाया जा रहा है",                            # Rice is being made from plastic
    "जियो दे रहा है मुफ्त में 1 साल का रिचार्ज",                    # Jio giving 1 year free recharge
    "गूगल दे रहा है फ्री में गिफ्ट वाउचर अपना जन्मदिन मनाने के लिए", # Google giving free gift vouchers for its birthday
]

for claim in hindi_wellknown_claims:
    matches = fact_check_search(claim)
    print(f"\nClaim: {claim}")
    if matches:
        for m in matches:
            print(f"  [{m['rating']}] {m['publisher']} — {m['url']}")
    else:
        print("  No matches.")


Claim: व्हाट्सएप अब मैसेज भेजने के लिए पैसे चार्ज करेगा
  No matches.

Claim: चावल प्लास्टिक से बनाया जा रहा है
  [False] Unknown — https://www.vishvasnews.com/politics/fact-check-video-of-production-of-plastic-beads-is-being-viral-as-plastic-rice/

Claim: जियो दे रहा है मुफ्त में 1 साल का रिचार्ज
  [False] विश्वास न्यूज — https://www.vishvasnews.com/viral/fact-check-jio-is-not-giving-one-year-free-recharge-on-new-year-viral-link-is-fake/

Claim: गूगल दे रहा है फ्री में गिफ्ट वाउचर अपना जन्मदिन मनाने के लिए
  No matches.
